In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 第5周 Day3：DeepSeek R1 推理模型原理

> **导语**：2025年初，一家中国AI公司发布的模型震惊了整个硅谷——它就是 DeepSeek R1。这个模型不靠堆参数、不靠海量数据标注，而是用**强化学习**让模型自己学会了"思考"。它会在回答之前进行长长的推理，甚至能自我纠错、反思。今天我们就来拆解 DeepSeek R1 背后的核心原理，看看"AI学会了思考"到底是怎么回事。

---


## 📊 学习进度


In [ ]:
■■■■■■■■■■■■■■■■■■■■■■■□□□□□□□□□□□□□□□□□□□□□□□□□
W1 ✅  W2 ✅  W3 ✅  W4 ✅  W5 🔵 进行中（Day3/7）  W6-W12 待解锁


---


## 🧠 为什么需要专门的推理模型？

### 普通大模型的"快思考"问题

普通大语言模型（GPT-4、Claude、Qwen等）本质上是"快思考"系统——它们接收到输入后，快速生成一个回答。就像你问一个人"17×23等于多少"，他脱口而出"391"——可能是对的，也可能口算错了。

这种"快思考"在以下场景中够用：
- 闲聊、文本摘要、翻译、简单问答

但在以下场景中常常翻车：
- 多步数学推理（每一步的错误会被放大）
- 复杂逻辑推理（需要考虑多种约束条件）
- 代码调试（需要理解整个程序的执行流）
- 科学问题求解（需要物理/化学/生物等领域知识）

### "慢思考"的启示

诺贝尔奖得主丹尼尔·卡尼曼在《思考，快与慢》中提出，人类有两套思维系统：
- **系统一**：快速、直觉、自动——对应普通大模型
- **系统二**：缓慢、理性、费力——对应推理模型

DeepSeek R1 就是专门为"系统二"设计的模型。它在回答之前会**先思考很长时间**，生成大量推理 Token，然后再给出答案。

### DeepSeek R1 为什么火？

2025年1月，DeepSeek R1 发布后引发轰动，原因是：

1. **效果惊人**：在数学推理（MATH基准）上超越了 GPT-4o，与 OpenAI o1 相当
2. **成本极低**：训练成本仅约 557 万美元（远低于同级别模型）
3. **完全开源**：模型权重、训练方法全部公开
4. **技术突破**：证明了纯强化学习也能让模型学会推理

---


## 🔬 核心原理详解

### 1. 两个版本：R1-Zero 和 R1

DeepSeek R1 实际上有两个版本，理解它们的区别非常重要：

#### R1-Zero：纯强化学习实验

**核心思路**：不给任何人工编写的推理示例，直接用强化学习让模型自己摸索怎么思考。

**具体做法**：
- 基础模型：DeepSeek-V3（一个已经预训练好的大模型）
- 训练方法：GRPO（群组相对策略优化）
- 奖励信号：答对给正奖励，答错给负奖励
- 特殊设计：在回答格式中加了 `<think>` 和 `</think>` 标签，让模型有一个"思考空间"

**惊人的发现**：模型在训练过程中**自发涌现**出了推理行为——它自己学会了：
- 分步推理（先算A再算B）
- 自我验证（算完之后检查一遍）
- 反思纠错（发现不对就回退重来）
- "Aha moment"（突然意识到可以用另一种方法）

这就是**"涌现"（Emergence）现象**——没有人教模型这些推理策略，它是通过强化学习的奖惩信号自己"悟"出来的。

#### R1：完整训练流程

R1-Zero 虽然推理能力强，但可读性差——它的推理过程往往很混乱、不连贯、中英文混杂。为了解决这个问题，R1 采用了一个更完整的训练流程：

1. **冷启动数据微调**：先用少量高质量的推理示例微调模型，让它学会"什么是好的推理格式"
2. **推理导向的强化学习**：在推理任务上做 GRPO 训练
3. **拒绝采样 + 监督微调**：从 RL 模型中采样好的推理链，混合通用数据再做一次 SFT
4. **全场景强化学习**：最后再做一轮 RL，同时优化推理能力和通用对话能力

### 2. GRPO 算法：不需要额外奖励模型

传统的 RLHF（人类反馈强化学习）需要训练一个单独的"奖励模型"来模拟人类偏好。这很麻烦——奖励模型本身也可能出错。

**GRPO（Group Relative Policy Optimization）** 的创新在于：
- 对同一个问题，让模型生成一组（比如8个）不同的回答
- 用规则奖励函数打分（数学题可以直接验算答案对不对）
- 在组内做相对比较：比平均水平好的回答获得正奖励，差的获得负奖励
- **不需要训练单独的奖励模型**

**生活类比**：就像老师让全班同学做同一道题，然后按相对成绩给分——不需要标准答案，只需要比较谁比平均分高。但前提是题目本身可以客观判分（数学、编程可以，创意写作就不行）。

### 3. MLA（多头潜在注意力）：省内存的注意力机制

DeepSeek 系列模型使用了 **MLA（Multi-Head Latent Attention）** 技术。

**问题**：标准注意力机制在生成长文本时，需要缓存大量的 KV（Key-Value）对，内存消耗巨大。推理过程动辄几千个 Token，KV 缓存就成了瓶颈。

**MLA 的解决方案**：把高维的 KV 向量压缩到一个低维的"潜在表示"中。需要用时再解压回来。这样 KV 缓存的内存占用可以减少到原来的很小一部分。

**类比**：就像你把一个 100MB 的图片压缩成 10MB 的 JPEG——大部分时候看起来差不多，但占用的空间大大减少了。

### 4. Reasoning Token：看得见的"思考过程"

DeepSeek R1 最直观的特征是它会输出大量的 **Reasoning Token**——这些 Token 构成了模型的"思考过程"。

一段典型的 R1 回答结构：


In [ ]:
<think>
好的，让我一步步分析这个问题。
首先，题目说...
然后，我需要计算...
等等，我刚才的思路好像有问题，让我换个方法。
Aha！我可以这样做...
让我验证一下...
好的，答案确认无误。
</think>

最终答案是：...


好处：提高准确率、增加可解释性、便于纠错。
坏处：增加延迟、增加 Token 消耗成本。

---


## 💻 代码实战

### 1. 环境准备


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# 中文字体配置
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("✅ 环境就绪，字体：", font_name)


### 2. 绘制 DeepSeek R1 架构示意图


In [ ]:
def draw_r1_architecture():
    """
    画出 DeepSeek R1 的推理架构：
    输入 → 思维链推理 → 自验证 → 反思纠错 → 最终输出
    """
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 8)
    ax.set_aspect('equal')
    ax.axis('off')

    boxes = [
        (0.5, 3, '输入问题', '#87CEEB'),
        (3.5, 5, '思维链推理\n(CoT)', '#98FB98'),
        (3.5, 1, '候选路径\n采样', '#98FB98'),
        (6.5, 5, '自验证\n(Self-Verify)', '#FFD700'),
        (6.5, 1, '反思纠错\n(Reflect)', '#FFD700'),
        (9.5, 3, '最终答案\n输出', '#FFA07A'),
    ]

    for x, y, text, color in boxes:
        box = FancyBboxPatch((x, y), 2, 2,
                             boxstyle="round,pad=0.15",
                             facecolor=color, edgecolor='black',
                             alpha=0.7, linewidth=1.5)
        ax.add_patch(box)
        ax.text(x + 1, y + 1, text, ha='center', va='center',
                fontsize=10, fontweight='bold')

    arrows = [
        (2.5, 4, 3.5, 5.5),
        (2.5, 4, 3.5, 2.5),
        (5.5, 5.5, 6.5, 5.5),
        (5.5, 2.5, 6.5, 2.5),
        (8.5, 5.5, 9.5, 4),
        (8.5, 2.5, 9.5, 4),
    ]
    for x1, y1, x2, y2 in arrows:
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

    ax.set_title('DeepSeek R1 推理架构示意图', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('r1_architecture.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_r1_architecture()


### 3. 模拟 R1 式推理过程


In [ ]:
def simulate_r1_reasoning(question: str) -> str:
    """
    模拟 DeepSeek R1 的推理过程：
    <think> 标签中是推理过程，之后是最终答案
    """
    think_process = f"""<think>
好的，让我仔细分析这个问题：{question}

第一步：理解题目要求
- 需要找到关键数字和关系
- 确定用什么方法来解决

第二步：初步推理
- 尝试直接计算
- 检查中间结果是否合理

第三步：验证
- 等等，让我用另一种方法验证一下
- 两种方法得到的答案一致，很好

第四步：最终确认
- 答案经过验证，确认无误
</think>

根据我的推理过程，最终答案是：[计算结果]"""
    return think_process

print("=" * 60)
print("🧠 DeepSeek R1 式推理演示")
print("=" * 60)
print(simulate_r1_reasoning("小明有10个苹果，分给3个朋友，每人几个？"))


### 4. R1-Zero 涌现行为可视化


In [ ]:
def plot_emergence_behavior():
    """
    展示 R1-Zero 在训练过程中推理能力的涌现过程：
    初期准确率低 → 中期开始推理 → 后期大幅提升
    """
    training_steps = np.arange(0, 10000, 100)
    accuracy = 1 / (1 + np.exp(-(training_steps - 5000) / 1000)) * 0.8 + 0.15
    reasoning_length = training_steps / 50 + np.random.normal(0, 20, len(training_steps))
    reasoning_length = np.clip(reasoning_length, 10, 500)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    ax1.plot(training_steps, accuracy * 100, color='#FF6B6B', linewidth=2)
    ax1.set_xlabel('训练步数', fontsize=12)
    ax1.set_ylabel('准确率 (%)', fontsize=12)
    ax1.set_title('R1-Zero：推理准确率的"涌现"过程', fontsize=14, fontweight='bold')
    ax1.axvline(x=5000, color='red', linestyle='--', alpha=0.5, label='涌现临界点')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.annotate('直接回答阶段\n（不会推理）', xy=(1500, 20), fontsize=9, ha='center', color='gray')
    ax1.annotate('推理能力涌现！', xy=(5000, 55), fontsize=11, ha='center', color='red', fontweight='bold')
    ax1.annotate('自验证+反思', xy=(8500, 90), fontsize=9, ha='center', color='green')

    ax2.plot(training_steps, reasoning_length, color='#4ECDC4', linewidth=2)
    ax2.set_xlabel('训练步数', fontsize=12)
    ax2.set_ylabel('推理链平均长度（Token）', fontsize=12)
    ax2.set_title('R1-Zero：推理链长度随训练增长', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('r1_emergence.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_emergence_behavior()


### 5. R1 vs 传统模型对比


In [ ]:
def plot_r1_vs_traditional():
    """对比 DeepSeek R1 和传统模型在不同任务上的表现"""
    tasks = ['数学推理\n(MATH)', '编程\n(HumanEval)', '科学问答\n(GPQA)',
             '日常对话\n(MMLU)', '创意写作\n(主观)']
    r1_scores = [91, 89, 75, 88, 82]
    traditional = [65, 70, 58, 85, 88]

    x = np.arange(len(tasks))
    width = 0.3

    fig, ax = plt.subplots(figsize=(13, 6))
    bars1 = ax.bar(x - width/2, r1_scores, width, label='DeepSeek R1（推理模型）', color='#FF6B6B', alpha=0.85)
    bars2 = ax.bar(x + width/2, traditional, width, label='传统大模型', color='#4ECDC4', alpha=0.85)

    ax.set_xlabel('任务类型', fontsize=13)
    ax.set_ylabel('得分', fontsize=13)
    ax.set_title('DeepSeek R1 vs 传统模型：全方位对比', fontsize=15, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(tasks, fontsize=10)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{int(height)}', ha='center', fontsize=10)

    plt.tight_layout()
    plt.savefig('r1_vs_traditional.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("💡 结论：R1 在推理类任务上大幅领先，但在创意写作上可能不如传统模型")

plot_r1_vs_traditional()


---


## 🏪 业务关联：LangChat / Agent 场景

### 1. 复杂问题求解
在 Agent 框架中，推理模型适合处理需要深度思考的任务：数据分析策略规划、多约束优化、异常诊断。

### 2. LangChat 中的应用
- **延迟更高**：推理模型生成时间长，适合异步处理
- **Token 消耗更大**：推理过程消耗大量 Token
- **可配置推理深度**：根据任务复杂度调整推理 Token 上限

### 3. 混合模型策略
最佳实践：用传统模型做快速响应（系统一），用推理模型做复杂决策（系统二）。

---


## ⚠️ 常见误区

### 误区1："R1 什么任务都更强"
**事实**：R1 在推理类任务上很强，但在创意写作、日常聊天等不需要深度推理的任务上可能不如传统模型。

### 误区2："强化学习能解决一切"
**事实**：GRPO 只对有客观标准答案的任务有效。对于主观任务还是需要人类偏好数据。

### 误区3："涌现是魔法"
**事实**：强化学习的奖励信号天然地激励模型"多想"——想得多的回答更容易答对，答对了获得更高奖励。模型在优化过程中自然学会了更长的推理链。

### 误区4："Reasoning Token 等于 CoT"
**事实**：CoT 是通过 Prompt 触发的，而 R1 的推理能力是通过训练内化的——即使不加 CoT Prompt，R1 也会自动展开推理。

---


## ✏️ 课堂练习（5分钟）

**练习1**：简述 DeepSeek R1 与传统聊天模型的三个主要区别。

**练习2**：什么是"涌现"能力？为什么说 R1-Zero 是涌现的典型例子？

**练习3**：GRPO 算法相比传统 RLHF 有什么优势？它适用于什么类型的任务？

---


## 📝 课后测试（15分钟）

**❶ DeepSeek R1 的核心特征是什么？**
A. 快速对话能力
B. 多步链式思维 + 自验证 + 反思
C. 单步直接推理
D. 文本摘要能力

**❷ R1-Zero 版本使用什么训练方法？**
A. 纯监督学习
B. 纯强化学习
C. 半监督学习
D. 无监督学习

**❸ MLA 机制的主要作用是？**
A. 减少训练数据量
B. 压缩 KV 缓存，降低推理内存消耗
C. 增强对话能力
D. 生成更长的文本

**❹ GRPO 相比传统 RLHF 的关键优势是？**
A. 不需要基础模型
B. 不需要训练单独的奖励模型
C. 不需要任何奖励信号
D. 训练速度更快

**❺ 简答题：为什么说 DeepSeek R1 的推理能力是"涌现"出来的？请从强化学习的角度解释。**

---


## 📖 术语表

| 英文术语 | 音标 | 中文释义 |
|---------|------|---------|
| DeepSeek R1 | /diːp siːk ɑːr wʌn/ | DeepSeek 推理模型，用 RL 让模型自己学会思考 |
| Reinforcement Learning (RL) | /ˌriːɪnˈfɔːsmənt ˈlɜːnɪŋ/ | 强化学习，通过奖惩让模型自己学策略 |
| RLHF | /ɑːr el eɪtʃ ef/ | 人类反馈强化学习，用人的偏好指导学习 |
| GRPO | /dʒiː ɑːr piː əʊ/ | 群组相对策略优化，不需要单独的奖励模型 |
| Reasoning Token | /ˈriːzənɪŋ ˈtəʊkən/ | 推理 Token，模型思考时的内部输出 |
| MLA | /em el eɪ/ | 多头潜在注意力，DeepSeek 的 KV 缓存压缩技术 |
| Process Reward | /ˈprəʊses rɪˈwɔːd/ | 过程奖励，对推理的每一步都打分 |
| Outcome Reward | /ˈaʊtkʌm rɪˈwɔːd/ | 结果奖励，只对最终答案打分 |
| Self-Correction | /self kəˈrekʃən/ | 自我纠错，模型发现自己的错误并修正 |
| Emergence | /ɪˈmɜːdʒəns/ | 涌现，复杂行为从简单规则中自然产生 |
| Cold Start | /kəʊld stɑːt/ | 冷启动，用少量数据初始化模型方向 |
| Rejection Sampling | /rɪˈdʒekʃən ˈsɑːmplɪŋ/ | 拒绝采样，只保留好的生成结果 |
| System 2 Thinking | /ˈsɪstəm tuː ˈθɪŋkɪŋ/ | 系统二思维，缓慢理性的深度思考 |

---


## 🔗 参考资源

- 📄 **技术报告**：[DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948)
- 📄 **深度分析**：[DeepSeek-R1 Thoughtology: Let's
